**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Channel Coding

[Digital Communications](./Digital_Communications.ipynb) got bits across a channel — *mostly*. [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) promised that rates below capacity can be error-**free**. This course builds the machinery that cashes that promise: Hamming codes, convolutional codes with Viterbi decoding (verified against brute force), and a look at the LDPC/polar codes in your phone.

## 1. Pre-requisites

- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) S4 (capacity).
- [Digital Communications](./Digital_Communications.ipynb) (the channel being protected).
- Binary arithmetic (XOR as addition mod 2).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Block Codes & Hamming* (~40 min)
**Goal:** add parity with structure: detect, then CORRECT errors; the Hamming (7,4) built from scratch.
**Feeds into:** Session 2 (convolutional codes & Viterbi).

---

## 2. Redundancy with Geometry

💡 **Intuition.** Repetition (send everything 3×) corrects single errors at rate 1/3 — brutal. Hamming's 1948 insight: parity bits can each *watch an overlapping subset* of data bits, so the pattern of failed checks (**the syndrome**) doesn't just announce an error — it spells out the *address* of the flipped bit. The (7,4) code corrects any single error at rate 4/7. Geometrically: codewords are spread out so every single-bit corruption still lands *nearest* its true codeword — minimum distance 3, correcting $\lfloor (d-1)/2 \rfloor = 1$ error.

In [ ]:
# Hamming (7,4): generator and parity-check matrices over GF(2)
# every single-bit error on every message must be corrected — exhaustive check

# YOUR CODE HERE


**What just happened.** All **112** cases — 16 possible messages × 7 possible single-bit error positions — corrected, with the `assert` making it a permanent guarantee rather than an observation.

Note the character of that check. It is not a simulation that ran many random trials and found no failures; it is an *exhaustive* enumeration of every single-error scenario the code claims to handle. Where a problem is small enough to check completely, checking it completely beats sampling it, and the claim "Hamming(7,4) corrects any single error" is here established rather than illustrated.

**The syndrome is an address, and that is the whole idea.** A lone parity bit announces that *something* is wrong. Hamming's insight was to have each parity bit watch an overlapping subset of the data bits, so the *pattern* of which checks fail identifies which bit flipped. Look at the decoder: `np.where((H.T == synd).all(1))` searches the syndrome among the columns of $H$ — because the syndrome produced by a single error at position $j$ **is** column $j$ of $H$. Detection becomes localisation, and localisation is correction. That is why the columns of $H$ are all distinct and non-zero; it is a design requirement, not an accident.

**The geometric reading.** Codewords are 16 points in $\{0,1\}^7$, placed so any two differ in at least 3 positions. Flip one bit and you sit at distance 1 from the true codeword and distance $\geq 2$ from every other — so nearest-neighbour decoding is unambiguous. Minimum distance $d$ corrects $\lfloor (d-1)/2\rfloor$ errors, giving 1 here. That formula also tells you what the code *cannot* do: two errors move you distance 2, which can be closer to a different codeword, and the decoder will then "correct" toward the wrong one — adding a third error while reporting success.

**And the efficiency is the point.** Repetition-by-3 also corrects one error, at rate 1/3. Hamming does it at rate **4/7** — nearly double the throughput for identical protection. The gain comes entirely from letting parity bits share responsibility rather than each guarding one bit alone. The next cell measures what that buys on a real channel.

In [ ]:
# BER on a binary symmetric channel: uncoded vs Hamming, simulated

# YOUR CODE HERE


**What just happened.** Two curves on log-log axes, and the coded one is *steeper*. That difference in slope, not the vertical gap, is the coding gain.

The exact BERs for this code are worth having in front of you:

| channel $p$ | uncoded | Hamming(7,4) | ratio |
|---|---|---|---|
| 0.001 | 0.0010 | 0.00001 | 0.01 |
| 0.003 | 0.0030 | 0.00008 | 0.03 |
| 0.01 | 0.0100 | 0.00087 | 0.09 |
| 0.03 | 0.0300 | 0.0074 | 0.25 |
| 0.1 | 0.1000 | 0.0669 | 0.67 |

**Why the slope is 2.** Uncoded BER is exactly $p$ — linear. The coded curve is close to $10p^2$ at small $p$ — quadratic. The reason is structural: every *single* error is corrected, so the code only fails when **two or more** bits flip in the same 7-bit block, and that has probability $\sim p^2$. Fixing single errors did not shift the curve down by a constant; it changed which event causes failure, and the new event is rarer by an order in $p$. At $p = 0.001$ that is a 100× improvement.

**And now the part that matters practically: the gain evaporates.** Read the ratio column downward — 0.01, 0.03, 0.09, 0.25, 0.67 — and extend it: at $p = 0.2$ the coded BER is 0.196 against an uncoded 0.200, essentially break-even. The code stops helping.

The mechanism is worth stating plainly, because it is not merely diminishing returns. When two errors land in one block, the syndrome points at some *third* position, and the decoder confidently flips a bit that was correct. At high noise the "corrector" is actively adding errors, and only the fact that it still fixes the (now less common) single-error cases keeps it near parity. **A code that corrects one error becomes worthless once two errors are typical** — and there is no warning in the output when you cross that line.

The general lesson: every code has a noise level beyond which it stops paying, and you cannot extrapolate a coding gain measured at low $p$ to a noisier channel. This is also why the rate matters — Hamming spends 3 redundant bits per 4 data bits, and if that redundancy buys nothing at your operating point, you have simply thrown away 43% of your throughput.

Session 2 attacks the same problem from a different direction: instead of protecting each block independently, entangle every bit with its neighbours so that errors have to defeat a whole *path* rather than a single block.

---
### 🕐 Session 2 of 3 — *Convolutional Codes & Viterbi* (~40 min)
**Goal:** encode with memory; decode with dynamic programming — verified against brute force.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (modern codes).

---

## 3. Codes with Memory

💡 **Intuition.** A convolutional encoder is an [FIR filter over GF(2)](./Foundations_of_Signal_Processing_1.ipynb): each input bit emits output bits that depend on the last $K$ inputs, entangling every bit with its neighbors. Decoding = finding the most likely *path* through the encoder's state **trellis** — and since path cost is additive, dynamic programming (**Viterbi**) finds the exact best path in linear time. It is [Bellman's principle](../Intro_Mach_Learn/Reinforcement_Learning.ipynb) applied to decoding — the same algorithm that powers speech recognition and DNA alignment.

In [ ]:
# rate-1/2, K=3 convolutional code (the classic [7,5] octal generators)
# ORACLE: Viterbi must equal brute-force maximum-likelihood over ALL 2^10 messages

# YOUR CODE HERE


**What just happened.** Two claims, and the first is the stronger one. **Viterbi's output equals the brute-force maximum-likelihood decode**, both landing at Hamming distance 2 from the received word. And separately, the decoded message equals the transmitted one, despite the channel flipping bits at 8%.

**Why the oracle matters more than the success.** Brute force enumerated all $2^{10} = 1024$ possible messages, encoded each, and picked whichever came closest to what was received — that is the definition of maximum-likelihood decoding on a binary symmetric channel, and it is optimal by construction. Viterbi visited 10 time steps × 4 states and got the *same answer*. So this is not evidence that Viterbi usually works; it is a check that a linear-time dynamic program reproduces an exponential-time optimum exactly. Verifying optimality rather than merely adequacy is a much stronger form of testing, and it is available here only because the problem is small enough to brute-force.

**The dynamic programming argument, in one line.** Path cost is **additive** across time steps. So once you know the cheapest path reaching a given state at time $t$, any costlier path into that same state can be discarded forever — it can never become part of a global optimum. That is [Bellman's principle](../Intro_Mach_Learn/Reinforcement_Learning.ipynb), and it collapses a search over $2^{10}$ paths into 40 small comparisons. The same algorithm decodes hidden Markov models in speech recognition and aligns DNA sequences; only the branch metric changes.

**Why the code can absorb 8% errors at all.** A convolutional encoder is an [FIR filter over GF(2)](./Foundations_of_Signal_Processing_1.ipynb) — each input bit influences several output bits, so the bits are entangled and cannot be judged locally. Hamming protected each 7-bit block independently, and two errors inside one block defeated it. Here an error must be bad enough to make an entirely *different path* through the trellis look more likely, and isolated flips rarely manage that. The failure mode moved from "two errors in a block" to "a burst long enough to fool a whole path."

**Two details worth noticing in the code.** The `+ [0, 0]` tail bits flush the encoder state so the trellis ends at a known state — without them the final message bits get less protection than the rest. And the branch metric is Hamming distance, i.e. *hard-decision* decoding: the receiver has already thresholded each sample to a bit before decoding. Real systems keep the analogue confidence and use soft decisions, which typically buys around 2 dB — a substantial amount, and free apart from arithmetic.

In [ ]:
# BER race at equal ENERGY per information bit (soft comparison, hard-decision decoding)

# YOUR CODE HERE


**What just happened.** The convolutional curve sits orders of magnitude below the uncoded line at moderate noise, and — as with Hamming — the interesting feature is that it is *steeper*, not merely lower.

**Why memory beats blocks.** Hamming's failure mode was two errors inside one 7-bit block, which happens with probability $\sim p^2$. A convolutional code has no blocks: each bit influences several outputs, so an error must be severe enough to make a *different path* through the trellis fit the received sequence better than the true one. Isolated flips almost never manage that, because the wrong path disagrees with the received bits in many other places too. Errors have to arrive in a coordinated burst to win, and that is far rarer than two independent flips.

**And the decoding is exactly optimal, which is unusual.** The previous cell verified Viterbi against exhaustive maximum likelihood, so this curve is not "a good decoder's performance" but *the best any decoder could do* for this code on this channel. Separating code design from decoder quality matters: a disappointing result could mean a weak code or a weak decoder, and here the decoder is provably not the limitation. Anything left on the table belongs to the code.

**Read the axes carefully before drawing conclusions.** The uncoded reference is BER $= p$ and the coded curve is plotted against the same channel flip probability — but the convolutional code is **rate 1/2**, so it transmits two channel bits per information bit. At a fixed transmit power that means less energy per channel bit and therefore a *higher* $p$ than the uncoded system would face. The comparison as plotted holds $p$ fixed rather than energy-per-information-bit, which flatters the code somewhat. The cell's comment flags this ("at equal energy per information bit... hard-decision decoding"); doing the comparison properly requires mapping $p$ to $E_b/N_0$ through the modulation. The qualitative conclusion survives — convolutional coding genuinely wins at moderate noise — but the exact gap is not the one a link-budget calculation would produce.

**What real systems add on top.** This is hard-decision decoding, with each sample thresholded to a bit before the decoder sees it. Keeping the analogue confidence and using soft branch metrics is worth roughly 2 dB, which is a large amount for a change that costs only arithmetic. Constraint length also buys performance: $K = 3$ has 4 states, while GPS uses $K = 7$ with 64 states, and complexity grows as $2^{K-1}$ while the gain keeps improving. That exponential cost is exactly why Session 3's LDPC and polar codes exist — they reach far closer to capacity without an exponentially growing decoder.

---
### 🕐 Session 3 of 3 — *Modern Codes at a Glance* (~30 min)
**Goal:** why LDPC and polar codes closed the gap to Shannon — mechanisms, not implementations.
**Builds on:** Session 2.

---

## 4. Closing the Last dB

> ℹ️ **Survey session** — mechanisms and intuition; production LDPC/polar decoders are a course of their own and are *not* implemented here.

💡 **Intuition (LDPC).** A *sparse* random parity-check matrix — each bit in a few checks, each check watching a few bits — decoded by **belief propagation**: bits and checks pass probability messages on the graph ([graph signal processing](./Graph_Signal_Processing.ipynb) territory) until consistent. Gallager invented them in 1962; they waited 35 years for hardware. Your Wi-Fi and 5G data channels run them within ~0.5 dB of Shannon's limit.

💡 **Intuition (polar).** Chain two-bit butterflies recursively ([FFT structure](./Foundations_of_Signal_Processing_1.ipynb)!) and channels *polarize*: some synthetic bit-channels become nearly perfect, others nearly useless. Put data on the good ones, zeros on the bad — the first codes *provably* achieving capacity (Arıkan, 2009). 5G control channels use them.

**The map:**

| | Hamming | Convolutional | LDPC | Polar |
|---|---|---|---|---|
| Decoding | syndrome table | Viterbi (exact DP) | belief propagation (iterative) | successive cancellation |
| Gap to capacity | far | moderate | ~0.5 dB | → 0 (provable) |
| Where | teaching, ECC RAM | GPS, legacy comms | Wi-Fi/5G data | 5G control |

## 5. Conclusion

Syndromes spell out error addresses; trellises make optimal decoding a shortest path (verified against brute force); sparsity + message passing and recursive polarization close the last decibels to Shannon. The promise of [Information Theory S4](../Intro_Math/Information_Theory/Information_Theory.ipynb) is now an engineering fact you've simulated.

---
## Where next

- [Digital Communications](./Digital_Communications.ipynb) — the modem these codes ride.
- [Graph Signal Processing](./Graph_Signal_Processing.ipynb) — belief propagation's playing field.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — real packets carrying real codes.